# 04 · Phone channel, simulated end to end

The call server is driven exactly as Twilio would drive it: HMAC-signed webhooks, a recording handed over, then polling for the result. Twilio's network is not involved. The recording download is replaced by a local audio file, so this runs offline with dummy credentials.

Live use: `scripts/start_call.ps1 -Number +1XXXXXXXXXX` (see the README).

In [1]:
import json, os, pathlib, time, warnings
import pandas as pd
warnings.filterwarnings("ignore")
if pathlib.Path.cwd().name == "notebooks":
    os.chdir("..")
os.environ.pop("ELEVENLABS_API_KEY", None)  # local Piper voice only: these notebooks demonstrate the offline core
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 200)

import base64, hashlib, hmac, re, shutil
os.environ.update(TWILIO_ACCOUNT_SID="ACnotebookdemo", TWILIO_AUTH_TOKEN="notebook-secret",
                  PUBLIC_URL="https://demo.example")
os.environ.pop("TWILIO_PHONE_NUMBER", None)

from IPython.display import Audio, display
from fastapi.testclient import TestClient
import coop_assistant.apps.call_server as cs      # loads Whisper and warms the Q&A model on import
from coop_assistant.speech import voice

In [2]:
question_wav, _ = voice.speak("Parchment coffee, grade A, in Nyeri. The buyer offered 105 shillings per kilo.", "en")

def fake_download(url):
    dst = cs.AUDIO_DIR / "in-notebook.wav"
    shutil.copy(question_wav, dst)
    return dst

cs._download, cs._delete_remote = fake_download, (lambda url: None)
client = TestClient(cs.app)

def twilio_post(path, form, sign=True):
    payload = "https://demo.example" + path + "".join(k + form[k] for k in sorted(form))
    sig = base64.b64encode(hmac.new(b"notebook-secret", payload.encode(), hashlib.sha1).digest()).decode()
    return client.post(path, data=form, headers={"X-Twilio-Signature": sig if sign else "forged"})

def show(label, response):
    print(f"--- {label} [{response.status_code}]")
    print(re.sub(r"(<(?:Play|Record|Redirect|Pause|Hangup)[^>]*>)", r"\n  \1", response.text), "\n")

### 1. A forged request is rejected

In [3]:
show("unsigned /voice", twilio_post("/voice", {"CallSid": "CA-demo"}, sign=False))

--- unsigned /voice [403]
{"detail":"bad signature"} 



### 2. The call is answered: greeting, then record the question

In [4]:
show("/voice", twilio_post("/voice", {"CallSid": "CA-demo"}))

--- /voice [200]
<?xml version="1.0" encoding="UTF-8"?><Response>
  <Play>https://demo.example/audio/greeting.wav</Play>
  <Record action="https://demo.example/turn" method="POST" maxLength="20" timeout="3" playBeep="true" finishOnKey="0#" trim="trim-silence"/>
  <Play>https://demo.example/audio/bye.wav</Play>
  <Hangup/></Response> 



### 3. The question arrives; the caller hears a hold prompt while the hub works

In [5]:
turn = twilio_post("/turn", {"CallSid": "CA-demo", "RecordingUrl": "https://api.twilio.com/demo/Recordings/RE1"})
show("/turn", turn)
job = re.search(r"/result/([0-9a-f]+)", turn.text).group(1)
start = time.time()
while True:
    result = twilio_post(f"/result/{job}", {"CallSid": "CA-demo"})
    if "Redirect" not in result.text:
        break
show(f"/result after {time.time() - start:.1f}s", result)
answer_file = re.search(r"/audio/([0-9a-f]+\.wav)", result.text).group(1)
display(Audio(str(cs.AUDIO_DIR / answer_file)))

--- /turn [200]
<?xml version="1.0" encoding="UTF-8"?><Response>
  <Play>https://demo.example/audio/hold.wav</Play>
  <Redirect method="POST">https://demo.example/result/0304bc194ba94b5da9255428e380bc8c</Redirect></Response> 



[0304bc] heard (en): 'Parchment coffee, grade A, Nyeri, the buyer offered 105 shillings per kilo' -> ANSWER


--- /result after 8.1s [200]
<?xml version="1.0" encoding="UTF-8"?><Response>
  <Play>https://demo.example/audio/aa85fc6ca7fe4b75bab792f4ee280bbc.wav</Play>
  <Play>https://demo.example/audio/again.wav</Play>
  <Record action="https://demo.example/turn" method="POST" maxLength="20" timeout="3" playBeep="true" finishOnKey="0#" trim="trim-silence"/>
  <Play>https://demo.example/audio/bye.wav</Play>
  <Hangup/></Response> 



### 4. Pressing 0 hands over to a person; path traversal is refused

In [6]:
show("press 0", twilio_post("/turn", {"CallSid": "CA-demo", "Digits": "0"}))
print("GET /audio/..%2Fsecret.wav ->", client.get("/audio/..%2Fsecret.wav").status_code)

--- press 0 [200]
<?xml version="1.0" encoding="UTF-8"?><Response>
  <Play>https://demo.example/audio/5ff26448221c46388d79659db9cdcc5e.wav</Play>
  <Play>https://demo.example/audio/bye.wav</Play>
  <Hangup/></Response> 

GET /audio/..%2Fsecret.wav -> 404
